# Group 13 — Rebalancing April 26, 2026

Second rebalancing of the systematic long-short momentum strategy.

| Parameter | Value |
|---|---|
| Signal date | April 23, 2026 (last close before execution) |
| Execution date | April 26, 2026 |
| Strategy | Risk-Adjusted Momentum: Score = R(12m−1m) / σ(60d) |
| Parameters | λ=1.76, q=0.93, sector cap=30%, min/max weight=1%/25% |

## 1. Configuration

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from datetime import datetime, timedelta

# ── Strategy parameters (identical to backtest) ─────────────────────────────
LAMBDA        = 1.76
Q_EXP         = 0.93
LONG_CAPITAL  = 1_000_000
SHORT_CAPITAL = 1_000_000
SECTOR_CAP    = 0.30
W_MIN, W_MAX  = 0.01, 0.25
FEE_PER_TRADE = 2.0

# ── Rebalancing dates ────────────────────────────────────────────────────────
REBALANCING_DATE = 'April 26, 2026'
SIGNAL_DATE      = '2026-04-23'   # last close used to compute signal

# ── Portfolio BEFORE this rebalancing (after March 26 rebalancing) ───────────
# Source: Open_Positions_Chart.csv downloaded April 23, 2026
CURRENT_LONGS = {
    'HOLX':  1990,   # Hologic
    'EA':      49,   # Electronic Arts
    'STX':     24,   # Seagate Technology
    'CIEN':    45,   # Ciena Corp
    'FIX':    169,   # Comfort Systems USA
    'LITE':   321,   # Lumentum Holdings
    'SATS':  2099,   # EchoStar Corp
    'WBD':   1469,   # Warner Bros Discovery (residual after March 26)
    'MU':      26,   # Micron Technology  (residual after March 26)
    'WDC':     33,   # Western Digital    (residual after March 26)
}

CURRENT_SHORTS = {
    'LULU':  1379,   # Lululemon Athletica
    'BRO':   2333,   # Brown & Brown
    'HPQ':  12779,   # HP Inc
    'PAYX':  2611,   # Paychex
    'IT':      66,   # Gartner
    'FDS':     51,   # FactSet Research
    'GDDY':   122,   # GoDaddy
    'ROP':     28,   # Roper Technologies
    'FISV':   176,   # Fiserv
    'CPRT':  1512,   # Copart
}

print(f'Rebalancing  : {REBALANCING_DATE}')
print(f'Signal date  : {SIGNAL_DATE}')
print(f'Longs  before: {len(CURRENT_LONGS)} — {list(CURRENT_LONGS.keys())}')
print(f'Shorts before: {len(CURRENT_SHORTS)} — {list(CURRENT_SHORTS.keys())}')

/Users/jessicabourdouxhe/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Rebalancing  : April 26, 2026
Signal date  : 2026-04-23
Longs  before: 10 — ['HOLX', 'EA', 'STX', 'CIEN', 'FIX', 'LITE', 'SATS', 'WBD', 'MU', 'WDC']
Shorts before: 10 — ['LULU', 'BRO', 'HPQ', 'PAYX', 'IT', 'FDS', 'GDDY', 'ROP', 'FISV', 'CPRT']


## 2. Data Download

We download **historical** data up to the signal date only — no look-ahead bias.

In [2]:
# S&P 500 universe and sector map
url = 'https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv'
sp500_info = pd.read_csv(url)
sp500_info['Symbol'] = sp500_info['Symbol'].str.replace('.', '-', regex=False)
sec_col    = 'GICS Sector' if 'GICS Sector' in sp500_info.columns else 'Sector'
sector_map = dict(zip(sp500_info['Symbol'], sp500_info[sec_col]))

# Download data strictly up to SIGNAL_DATE (historical, no look-ahead)
end_dt    = datetime.strptime(SIGNAL_DATE, '%Y-%m-%d')
start_dt  = end_dt - timedelta(days=430)   # ~14 months for 252-day lookback
end_str   = end_dt.strftime('%Y-%m-%d')
start_str = start_dt.strftime('%Y-%m-%d')

print(f'Downloading data from {start_str} to {end_str}...')
tickers = sp500_info['Symbol'].tolist()
raw = yf.download(tickers, start=start_str, end=end_str, auto_adjust=True)['Close']
raw = raw.ffill().dropna(axis=1, thresh=int(len(raw) * 0.8))
print(f'Universe: {raw.shape[1]} stocks | {raw.shape[0]} trading days')
print(f'Last date in data: {raw.index[-1].date()}')

[*********************100%***********************]  503 of 503 completed

Universe: 502 stocks | 296 trading days
Last date in data: 2026-04-22


## 3. Signal Generation

$$\text{Score}_i = \frac{R_i^{12m-1m}}{\hat{\sigma}_i^{60d}}$$

Top 10 → **Long leg**. Bottom 10 → **Short leg**.

In [3]:
def compute_signals(price_df):
    """Risk-Adjusted Momentum scores for all stocks in price_df."""
    if len(price_df) < 252:
        raise ValueError(f'Need ≥252 days, got {len(price_df)}')
    mom   = (price_df.iloc[-21] / price_df.iloc[-252]) - 1
    vol   = price_df.tail(61).pct_change().dropna().std() * np.sqrt(252)
    score = (mom / vol).replace([np.inf, -np.inf], np.nan).dropna()
    return score.sort_values()

scores = compute_signals(raw)

target_longs  = list(scores.tail(10).index)   # highest score → long
target_shorts = list(scores.head(10).index)   # lowest  score → short

print(f'\n=== SIGNAL OUTPUT ({SIGNAL_DATE}) ===')
print('\nTop 10 LONGS:')
for t in target_longs:
    print(f'  {t:<8} score={scores[t]:+.3f}  sector={sector_map.get(t, "N/A")}')
print('\nBottom 10 SHORTS:')
for t in target_shorts:
    print(f'  {t:<8} score={scores[t]:+.3f}  sector={sector_map.get(t, "N/A")}')


=== SIGNAL OUTPUT (2026-04-23) ===

Top 10 LONGS:
  TER      score=+4.959  sector=Information Technology
  FIX      score=+5.757  sector=Industrials
  STX      score=+6.017  sector=Information Technology
  MU       score=+6.380  sector=Information Technology
  SATS     score=+7.134  sector=Communication Services
  CIEN     score=+8.071  sector=Information Technology
  WDC      score=+9.084  sector=Information Technology
  LITE     score=+14.444  sector=Information Technology
  WBD      score=+17.047  sector=Communication Services
  SNDK     score=+22.852  sector=Information Technology

Bottom 10 SHORTS:
  FISV     score=-1.836  sector=Financials
  CPRT     score=-1.569  sector=Industrials
  BRO      score=-1.288  sector=Financials
  GIS      score=-1.198  sector=Consumer Staples
  CPB      score=-1.174  sector=Consumer Staples
  ERIE     score=-1.147  sector=Financials
  PAYX     score=-1.092  sector=Industrials
  INVH     score=-1.070  sector=Real Estate
  KMB      score=-1.053  sect

## 4. Portfolio Optimization

$$\max_w \; U = E[R_p] - \frac{\lambda}{2} \cdot (\sigma_p^2)^q \quad (\lambda=1.76,\; q=0.93)$$

Constraints: weights sum to 1, each weight ∈ [1%, 25%], sector ≤ 30%.

In [4]:
def optimize_leg(tickers, price_df, capital, is_short=False):
    """
    Maximize U = E[R] - (lambda/2)*(sigma^2)^q.
    is_short=True inverts the return sign (we want high negative return).
    Returns: weights (sum=1), dollar allocations.
    """
    rets = price_df[tickers].tail(61).pct_change().dropna()
    mu   = rets.mean() * 252
    cov  = rets.cov()  * 252
    n    = len(tickers)
    sign = -1 if is_short else 1

    def neg_utility(w):
        p_ret = sign * float(np.dot(mu, w))
        p_var = float(np.dot(w, np.dot(cov, w)))
        p_var = max(p_var, 1e-10)              # numerical guard
        return -(p_ret - (LAMBDA / 2) * (p_var ** Q_EXP))  # correct utility

    # Constraints
    secs = [sector_map.get(t, 'Other') for t in tickers]
    cons = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    for s in set(secs):
        idx = [i for i, sec in enumerate(secs) if sec == s]
        cons.append({
            'type': 'ineq',
            'fun' : lambda w, idx=idx: SECTOR_CAP - np.sum(np.array(w)[idx])
        })

    # Sector-aware initial weights (avoids immediate constraint violation)
    sector_counts = {}
    for t in tickers:
        s = sector_map.get(t, 'Other')
        sector_counts[s] = sector_counts.get(s, 0) + 1
    w0 = np.array([SECTOR_CAP / sector_counts[sector_map.get(t, 'Other')]
                   for t in tickers])
    w0 = np.clip(w0, W_MIN, W_MAX)
    w0 = w0 / w0.sum()

    res = minimize(neg_utility, w0, method='SLSQP',
                   bounds=[(W_MIN, W_MAX)] * n,
                   constraints=cons,
                   options={'ftol': 1e-9, 'maxiter': 1000})

    weights = res.x if res.success else w0
    weights = np.clip(weights, W_MIN, W_MAX)
    weights = weights / weights.sum()
    return weights, weights * capital

w_long,  d_long  = optimize_leg(target_longs,  raw, LONG_CAPITAL,  is_short=False)
w_short, d_short = optimize_leg(target_shorts, raw, SHORT_CAPITAL, is_short=True)

latest_prices = raw.iloc[-1]
target_long_shares  = {t: int(d_long[i]  / latest_prices[t]) for i, t in enumerate(target_longs)}
target_short_shares = {t: int(d_short[i] / latest_prices[t]) for i, t in enumerate(target_shorts)}

print('=' * 65)
print('   TARGET LONG POSITIONS — April 26')
print('=' * 65)
print(f'{"Ticker":<8} {"Weight":>8} {"$ Amount":>12} {"Shares":>8}  Sector')
print('-' * 65)
for i, t in enumerate(target_longs):
    print(f'{t:<8} {w_long[i]:>7.1%} {d_long[i]:>12,.0f} {target_long_shares[t]:>8}  {sector_map.get(t, "N/A")}')
print(f'\nTotal long $: {sum(d_long):>,.0f}')

print('\n' + '=' * 65)
print('   TARGET SHORT POSITIONS — April 26')
print('=' * 65)
print(f'{"Ticker":<8} {"Weight":>8} {"$ Amount":>12} {"Shares":>8}  Sector')
print('-' * 65)
for i, t in enumerate(target_shorts):
    print(f'{t:<8} {w_short[i]:>7.1%} {d_short[i]:>12,.0f} {target_short_shares[t]:>8}  {sector_map.get(t, "N/A")}')
print(f'\nTotal short $: {sum(d_short):>,.0f}')

   TARGET LONG POSITIONS — April 26
Ticker     Weight     $ Amount   Shares  Sector
-----------------------------------------------------------------
TER         5.3%       52,747      136  Information Technology
FIX        26.2%      261,538      151  Industrials
STX         5.3%       52,747       90  Information Technology
MU          5.3%       52,747      108  Information Technology
SATS       18.5%      184,615     1508  Communication Services
CIEN        5.3%       52,747      105  Information Technology
WDC         5.3%       52,747      135  Information Technology
LITE        5.3%       52,747       60  Information Technology
WBD        18.5%      184,615     6755  Communication Services
SNDK        5.3%       52,747       53  Information Technology

Total long $: 1,000,000

   TARGET SHORT POSITIONS — April 26
Ticker     Weight     $ Amount   Shares  Sector
-----------------------------------------------------------------
FISV        1.0%       10,000      158  Financials
CPR

## 5. Trade Orders

Delta = target − current. Only non-zero deltas require a trade.

In [5]:
def print_orders(current_dict, target_dict, leg='LONG'):
    all_tickers = sorted(set(current_dict) | set(target_dict))
    orders = []
    for t in all_tickers:
        curr  = current_dict.get(t, 0)
        tgt   = target_dict.get(t, 0)
        delta = tgt - curr
        if delta == 0:
            action = 'HOLD'
        elif leg == 'LONG':
            action = f'BUY  +{delta:,}' if delta > 0 else f'SELL {delta:,}'
        else:  # SHORT
            if tgt == 0:
                action = f'COVER (close) {curr:,}'
            elif curr == 0:
                action = f'SHORT {tgt:,}'
            elif delta > 0:
                action = f'SHORT more +{delta:,}'
            else:
                action = f'COVER partial {-delta:,}'
        orders.append((t, curr, tgt, delta, action))
    return orders

long_orders  = print_orders(CURRENT_LONGS,  target_long_shares,  leg='LONG')
short_orders = print_orders(CURRENT_SHORTS, target_short_shares, leg='SHORT')

print('=' * 65)
print(f'   TRADE ORDERS — {REBALANCING_DATE} — LONG LEG')
print('=' * 65)
for t, curr, tgt, delta, action in long_orders:
    marker = '→ ' if delta != 0 else '  '
    price  = latest_prices.get(t, float('nan'))
    print(f'{marker}{t:<8} current={curr:>6,}  target={tgt:>6,}  | {action:<30} price≈${price:.2f}')

print()
print('=' * 65)
print(f'   TRADE ORDERS — {REBALANCING_DATE} — SHORT LEG')
print('=' * 65)
for t, curr, tgt, delta, action in short_orders:
    marker = '→ ' if delta != 0 else '  '
    price  = latest_prices.get(t, float('nan'))
    print(f'{marker}{t:<8} current={curr:>6,}  target={tgt:>6,}  | {action:<30} price≈${price:.2f}')

n_trades = sum(1 for *_, d, _ in long_orders + short_orders if d != 0)
print(f'\nTotal trades    : {n_trades}')
print(f'Transaction cost: {n_trades} × $2 = ${n_trades * FEE_PER_TRADE:.0f}')

   TRADE ORDERS — April 26, 2026 — LONG LEG
→ CIEN     current=    45  target=   105  | BUY  +60                       price≈$498.97
→ EA       current=    49  target=     0  | SELL -49                       price≈$202.78
→ FIX      current=   169  target=   151  | SELL -18                       price≈$1724.49
→ HOLX     current= 1,990  target=     0  | SELL -1,990                    price≈$nan
→ LITE     current=   321  target=    60  | SELL -261                      price≈$873.60
→ MU       current=    26  target=   108  | BUY  +82                       price≈$487.48
→ SATS     current= 2,099  target= 1,508  | SELL -591                      price≈$122.36
→ SNDK     current=     0  target=    53  | BUY  +53                       price≈$979.07
→ STX      current=    24  target=    90  | BUY  +66                       price≈$579.88
→ TER      current=     0  target=   136  | BUY  +136                      price≈$385.18
→ WBD      current= 1,469  target= 6,755  | BUY  +5,286             

## 6. Sector Compliance Audit

In [6]:
def sector_audit(tickers, weights, label):
    df = pd.DataFrame({'Ticker': tickers, 'Weight': weights,
                       'Sector': [sector_map.get(t, 'Unknown') for t in tickers]})
    by_sector = df.groupby('Sector')['Weight'].sum().sort_values(ascending=False)
    print(f'  {label}')
    print('  ' + '-'*45)
    ok = True
    for sec, w in by_sector.items():
        flag = ' ⚠ VIOLATION' if w > SECTOR_CAP + 0.001 else ''
        if flag:
            ok = False
        print(f'  {sec:<35} {w:.1%}{flag}')
    status = '✓ OK' if ok else '✗ SECTOR CAP EXCEEDED'
    print(f'  → {status}\n')
    return ok

print('=' * 55)
print('   SECTOR AUDIT')
print('=' * 55)
long_ok  = sector_audit(target_longs,  w_long,  'LONG LEG')
short_ok = sector_audit(target_shorts, w_short, 'SHORT LEG')

print('=' * 55)
print('   COMPLIANCE SUMMARY')
print('=' * 55)
print(f'  Long  positions : {len(target_longs):2d}/10 minimum  {"✓" if len(target_longs)>=10 else "✗"}')
print(f'  Short positions : {len(target_shorts):2d}/10 minimum  {"✓" if len(target_shorts)>=10 else "✗"}')
print(f'  Long  sector cap: {"✓" if long_ok  else "✗ VIOLATION — see note below"}')
print(f'  Short sector cap: {"✓" if short_ok else "✗ VIOLATION — see note below"}')
print()
if not long_ok or not short_ok:
    print('  NOTE: A sector violation indicates that the top-10 signal selection')
    print('  contains too many stocks from the same sector, making the 30% cap')
    print('  infeasible within the optimizer. In production, a pre-screening step')
    print('  would limit same-sector stocks before passing to the optimizer.')

   SECTOR AUDIT
  LONG LEG
  ---------------------------------------------
  Communication Services              36.9% ⚠ VIOLATION
  Information Technology              36.9% ⚠ VIOLATION
  Industrials                         26.2%
  → ✗ SECTOR CAP EXCEEDED

  SHORT LEG
  ---------------------------------------------
  Financials                          30.0%
  Consumer Staples                    30.0%
  Industrials                         30.0%
  Information Technology              9.0%
  Real Estate                         1.0%
  → ✓ OK

   COMPLIANCE SUMMARY
  Long  positions : 10/10 minimum  ✓
  Short positions : 10/10 minimum  ✓
  Long  sector cap: ✗ VIOLATION — see note below
  Short sector cap: ✓

  NOTE: A sector violation indicates that the top-10 signal selection
  contains too many stocks from the same sector, making the 30% cap
  infeasible within the optimizer. In production, a pre-screening step
  would limit same-sector stocks before passing to the optimizer.
